# Fase 3 — Memória: contexto da conversa e persistência entre sessões

**Problema desta fase:** a secretária precisa lembrar do que foi dito na conversa ("e o saldo dele?"), continuar depois de reiniciar e **não misturar** usuários nem conversas.

**Conceitos da disciplina demonstrados:** histórico passado em `messages` ao `invoke` (o padrão atual equivalente ao `RunnableWithMessageHistory`), janela curta, recortes antigos recuperados (busca lexical e FAISS com embeddings), memória longa de preferências.

**Decisões principais**
- Histórico persistido em **SQLite** por empresa, usuário e sessão; `/nova` (ou "Nova conversa") inicia sessão vazia sem apagar a anterior.
- Turnos que saem da janela de seis viram recortes; a busca **filtra por usuário e sessão antes** de pontuar (FAISS temporário só sobre o que é permitido).
- Trechos recuperados entram como contexto **não confirmado**; preferências só mudam por comando explícito.

**Evidências reais (fora deste notebook):** [falha lexical "cleinte"](../evidencias/fase3_falha_lexical_cleinte.json), [reteste com FAISS](../evidencias/fase3_faiss_reteste_manual.json), [troca de projeto "e o saldo dele?"](../evidencias/fase3_troca_projeto_reteste_manual.json). Guia: [memoria_fase3.md](../docs/memoria_fase3.md).

> **Fixtures:** respostas do agente e vetores de embedding são fixtures manuais. O notebook prova persistência, janela e isolamento; não prova compreensão do LLM.

Notebooks de origem: `incrementos/06_memoria_conversacional.ipynb`, `incrementos/07_resumos_recuperacao.ipynb`, `incrementos/08_preferencias_confirmadas.ipynb`, `incrementos/09_recuperacao_vetorial_memoria.ipynb`.

---
*Origem desta parte: `incrementos/06_memoria_conversacional.ipynb`.*

## Fase 3 — memória conversacional

Este notebook usa o mesmo serviço Python do agente. O teste local prova persistência e isolamento; **não demonstra compreensão por LLM**. A conversa real com o agente requer API e deve ser registrada separadamente.


In [1]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.memoria import MemoriaConversas


### 1. Uma conversa, três turnos e janela curta
Cada turno completo é salvo somente depois que a resposta existe. A janela de dois turnos limita o contexto enviado ao modelo.


In [2]:
pasta = tempfile.TemporaryDirectory()
caminho = Path(pasta.name) / 'memoria.sqlite3'
ana, bruno = Contexto('demo', 'ana'), Contexto('demo', 'bruno')
with Banco(caminho) as banco:
    banco.preparar_demo()
    memoria = MemoriaConversas(banco)
    sessao = memoria.iniciar(ana)
    memoria.registrar(ana, sessao, 'Tenho reunião com Alfa', 'Entendi; qual data?')
    memoria.registrar(ana, sessao, 'Amanhã às 14h', 'Preciso da data absoluta para agendar.')
    memoria.registrar(ana, sessao, 'Qual cliente mencionei?', 'Você mencionou Alfa.')
    janela = memoria.mensagens(ana, sessao, limite=2)
    assert len(janela) == 4 and janela[0]['content'] == 'Amanhã às 14h'
    print(janela)


[{'role': 'user', 'content': 'Amanhã às 14h'}, {'role': 'assistant', 'content': 'Preciso da data absoluta para agendar.'}, {'role': 'user', 'content': 'Qual cliente mencionei?'}, {'role': 'assistant', 'content': 'Você mencionou Alfa.'}]


> **Como ler o resultado:**
>
> - Cada pergunta e resposta fica salva; só as mais recentes vão para a IA (a **janela**).
> - Aqui a janela é de 2 trocas, só para a demonstração ficar curta. No programa são **6 trocas** (12 mensagens).

### 2. Reabrir, separar usuários e começar do zero
Reabrir o banco recupera os turnos. `/nova` no terminal encerra a sessão atual e cria outra vazia; o histórico antigo permanece auditável. O usuário é de demonstração, sem autenticação real.


In [3]:
with Banco(caminho) as banco:
    memoria = MemoriaConversas(banco)
    assert memoria.ultima_ativa(ana) == sessao
    assert len(memoria.ultimos(ana, sessao)) == 3
    assert memoria.ultima_ativa(bruno) is None
    try:
        memoria.ultimos(bruno, sessao)
    except PermissionError:
        print('Acesso de outro usuário negado, como esperado.')
    else:
        raise AssertionError('Falha de isolamento')
    memoria.encerrar(ana, sessao)
    nova = memoria.iniciar(ana)
    assert memoria.ultimos(ana, nova) == []
    print('Nova conversa vazia:', nova)
pasta.cleanup()


Acesso de outro usuário negado, como esperado.
Nova conversa vazia: f1812610b2a543eeb871d2342127c20b


> **Como ler o resultado:**
>
> - `Acesso de outro usuário negado`: um usuário não lê a conversa do outro.
> - `Nova conversa vazia`: conversa nova começa do zero; a antiga fica guardada.

---
*Origem desta parte: `incrementos/07_resumos_recuperacao.ipynb`.*

## Fase 3 — janela curta e recortes antigos

O mesmo serviço Python do menu 9 conserva seis turnos recentes. Quando um turno sai da janela, grava um **resumo extrativo**: a fala do usuário, sem a resposta do modelo. A busca abaixo usa palavras em comum, não embeddings nem FAISS. Os textos são fixtures locais; este notebook não prova compreensão por LLM.


In [4]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.memoria import MemoriaConversas


### 1. Um dado sai da janela
As respostas abaixo são apenas placeholders de teste. O recorte conserva a frase original do usuário e seu ID de turno, sem inferir que Alfa seja um cadastro válido.


In [5]:
pasta = tempfile.TemporaryDirectory()
caminho = Path(pasta.name) / 'resumos.sqlite3'
ana, bruno = Contexto('demo', 'ana'), Contexto('demo', 'bruno')
with Banco(caminho) as banco:
    banco.preparar_demo()
    memoria = MemoriaConversas(banco)
    sessao = memoria.iniciar(ana)
    memoria.registrar(ana, sessao, 'Estou falando do cliente Alfa.', '[FIXTURE LOCAL] Entendi.')
    for numero in range(6):
        memoria.registrar(ana, sessao, f'Assunto {numero}', '[FIXTURE LOCAL] Resposta.')
    janela = memoria.mensagens(ana, sessao)
    assert len(janela) == 12 and 'Alfa' not in str(janela)
    recuperados = memoria.buscar_resumos(ana, sessao, 'Qual cliente mencionei?')
    assert recuperados[0]['resumo_extrativo'] == 'Estou falando do cliente Alfa.'
    print('Janela:', len(janela), 'mensagens; recortes:', recuperados)


Janela: 12 mensagens; recortes: [{'turno_id': 1, 'resumo_extrativo': 'Estou falando do cliente Alfa.', 'pontuacao_lexical': 2, 'tipo_correspondencia': 'exata'}]


> **Como ler o resultado:**
>
> - `Janela: 12 mensagens`: as 6 últimas trocas.
> - `recortes`: o que saiu da janela não se perde; vira um **recorte** que pode ser buscado (memória episódica).

### 2. Reabrir e separar conversas
A busca verifica empresa, usuário e ID da sessão antes da pontuação lexical. `/nova` cria outra sessão sem reutilizar os recortes anteriores.


In [6]:
with Banco(caminho) as banco:
    memoria = MemoriaConversas(banco)
    assert memoria.buscar_resumos(ana, sessao, 'Qual cliente?')
    try:
        memoria.buscar_resumos(bruno, sessao, 'Qual cliente?')
    except PermissionError:
        print('Outro usuário bloqueado, como esperado.')
    else:
        raise AssertionError('Falha de isolamento')
    memoria.encerrar(ana, sessao)
    nova = memoria.iniciar(ana)
    assert memoria.buscar_resumos(ana, nova, 'Qual cliente?') == []
    print('Nova conversa sem recortes:', nova)
pasta.cleanup()


Outro usuário bloqueado, como esperado.
Nova conversa sem recortes: c1eb0c5b68954351a4a721416de8e2b5


> **Como ler o resultado:**
>
> - Recortes também são separados por usuário e por conversa.

### 3. Falha real e correção local
Renan escreveu `cleinte alfa` e depois perguntou por `cliente`; a busca exata retornou zero recortes. O teste abaixo reproduz o erro de digitação e mostra a correspondência aproximada. O notebook não substitui o reteste da resposta real do agente.


In [7]:
from secretaria.roteamento import roteador_pedido
with tempfile.TemporaryDirectory() as pasta_teste:
    with Banco(Path(pasta_teste) / 'tipos.sqlite3') as banco:
        banco.preparar_demo()
        memoria = MemoriaConversas(banco)
        sessao_teste = memoria.iniciar(ana)
        memoria.registrar(ana, sessao_teste, 'Estou falando do cleinte alfa', '[FIXTURE LOCAL] Entendi.')
        for numero in range(6):
            memoria.registrar(ana, sessao_teste, f'Oi {numero}', '[FIXTURE LOCAL] Olá.')
        achados = memoria.buscar_resumos(ana, sessao_teste, 'qual cliente eu mencionei agora pouco?')
        assert achados[0]['tipo_correspondencia'] == 'aproximada'
        assert roteador_pedido.invoke({'pergunta': 'Estou falando do cleinte alfa'})['rota'] == 'cliente'
        print(achados)


[{'turno_id': 1, 'resumo_extrativo': 'Estou falando do cleinte alfa', 'pontuacao_lexical': 1, 'tipo_correspondencia': 'aproximada'}]


> **Como ler o resultado:**
>
> - **Falha real corrigida**: "cleinte" (erro de digitação) não achava "cliente". Agora aceita palavras parecidas (`'aproximada'`).

---
*Origem desta parte: `incrementos/08_preferencias_confirmadas.ipynb`.*

## Fase 3 — preferências confirmadas entre sessões

Este notebook usa o mesmo serviço do menu 9, sem LLM. As preferências são comandos explícitos do usuário, não inferências do chat. O usuário de demonstração não é autenticação real.


In [8]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.preferencias import PreferenciasUsuario


### Gravar e corrigir
A versão cresce apenas quando o valor muda. A resposta do agente não altera esta memória.


In [9]:
pasta = tempfile.TemporaryDirectory()
arquivo = Path(pasta.name) / 'memoria.sqlite3'
ana, bruno = Contexto('demo', 'ana'), Contexto('demo', 'bruno')
with Banco(arquivo) as banco:
    banco.preparar_demo()
    preferencias = PreferenciasUsuario(banco)
    assert preferencias.salvar(ana, 'estilo', 'curto')['versao'] == 1
    assert preferencias.salvar(ana, 'estilo', 'detalhado')['versao'] == 2
    assert preferencias.listar(bruno) == []
    print('Ana:', preferencias.listar(ana), '| Bruno:', preferencias.listar(bruno))


Ana: [{'chave': 'estilo', 'valor': 'detalhado', 'versao': 2}] | Bruno: []


> **Como ler o resultado:**
>
> - `Ana: estilo = detalhado, versao 2`: a Ana mudou a preferência de "curto" para "detalhado". O Bruno não tem nada.
> - Preferência só é gravada quando o usuário pede (`/lembrar estilo=curto`, no terminal).

### Reinício, isolamento e esquecimento
Reabrir o SQLite conserva a preferência atual. Apagar elimina também as versões antigas dessa chave.


In [10]:
with Banco(arquivo) as banco:
    preferencias = PreferenciasUsuario(banco)
    assert preferencias.listar(ana)[0]['valor'] == 'detalhado'
    assert preferencias.listar(bruno) == []
    assert preferencias.apagar(ana, 'estilo')['estado'] == 'apagada'
    assert preferencias.listar(ana) == []
    total = banco.con.execute("SELECT COUNT(*) FROM historico_preferencias WHERE usuario_id=? AND chave=?", ('ana', 'estilo')).fetchone()[0]
    assert total == 0
    print('Persistência e exclusão verificadas; histórico remanescente:', total)
pasta.cleanup()


Persistência e exclusão verificadas; histórico remanescente: 0


> **Como ler o resultado:**
>
> - Continua depois de reabrir; e quando apaga, apaga de verdade, inclusive o histórico (`histórico remanescente: 0`).

### Teste no terminal
Execute `python executar.py --usuario ana`, abra a opção 9 e use `/lembrar estilo=detalhado`, `/memorias`, `/nova` e `/memorias`. Reinicie e confira novamente. Depois use `/esquecer-memoria estilo`. Sem API, esses comandos locais funcionam; a influência na resposta do modelo exige teste real separado.


---
*Origem desta parte: `incrementos/09_recuperacao_vetorial_memoria.ipynb`.*

## Fase 3 — FAISS nos recortes antigos

O menu 9 consulta embeddings OpenAI para recortes antigos da mesma conversa e monta um índice FAISS temporário após filtrar empresa, usuário e sessão no SQLite. Este notebook substitui a API por vetores de **fixture manual** para testar o fluxo sem custo; ele não mede a qualidade semântica do modelo real.


In [11]:
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.memoria import MemoriaConversas


### Vetores controlados para uma consulta reproduzível
Nesta fixture, `cliente` e `empresa` ocupam a mesma dimensão por definição manual. Em uso real, o modelo de embeddings determina as proximidades.


In [12]:
class EmbeddingsFixture:
    def vetor(self, texto):
        texto = texto.lower()
        return [1.0 if 'cliente' in texto or 'empresa' in texto else 0.0,
                1.0 if 'orçamento' in texto else 0.0, 0.1]
    def embed_documents(self, textos):
        return [self.vetor(texto) for texto in textos]
    def embed_query(self, texto):
        return self.vetor(texto)


In [13]:
with tempfile.TemporaryDirectory() as pasta:
    caminho = Path(pasta) / 'memoria.sqlite3'
    ana, bruno = Contexto('demo','ana'), Contexto('demo','bruno')
    with Banco(caminho) as banco:
        banco.preparar_demo()
        memoria = MemoriaConversas(banco)
        sessao = memoria.iniciar(ana)
        memoria.registrar(ana,sessao,'O cliente é Alfa.','[FIXTURE] Entendi.')
        memoria.registrar(ana,sessao,'O orçamento é Beta.','[FIXTURE] Entendi.')
        for numero in range(6):
            memoria.registrar(ana,sessao,f'Assunto {numero}','[FIXTURE] Entendi.')
        achados = memoria.buscar_resumos_vetoriais(ana,sessao,'Qual empresa foi citada?',EmbeddingsFixture(),limite=1)
        assert achados[0]['resumo_extrativo'] == 'O cliente é Alfa.'
        print('FAISS:',achados)
        try:
            memoria.buscar_resumos_vetoriais(bruno,sessao,'Qual empresa?',EmbeddingsFixture())
        except PermissionError:
            print('Outro usuário bloqueado')
        else:
            raise AssertionError('Falha de isolamento')
    with Banco(caminho) as banco:
        memoria = MemoriaConversas(banco)
        assert memoria.buscar_resumos_vetoriais(ana,sessao,'Qual empresa?',EmbeddingsFixture(),limite=1)
        nova = memoria.iniciar(ana)
        assert memoria.buscar_resumos_vetoriais(ana,nova,'Qual empresa?',EmbeddingsFixture()) == []
        print('Reinício preserva recortes; nova conversa não reutiliza recortes antigos.')


FAISS: [{'turno_id': 1, 'resumo_extrativo': 'O cliente é Alfa.', 'similaridade': 1.0, 'estrategia': 'faiss_embeddings'}]
Outro usuário bloqueado
Reinício preserva recortes; nova conversa não reutiliza recortes antigos.


> **Como ler o resultado:**
>
> - `faiss_embeddings` com `similaridade 1.0`: achou o recorte certo **por significado** (busca semântica).
> - Antes de buscar, filtra só os recortes do usuário e da conversa.
> - Vetores fixos aqui (sem API); no programa os embeddings vêm da OpenAI (`text-embedding-3-small`).

### Limites
O índice FAISS é reconstruído em RAM; o SQLite guarda os recortes. O menu 9 usa `text-embedding-3-small` quando a API está disponível e mostra a estratégia usada. Se a API falhar, usa a busca lexical local. Ainda é necessário um teste manual da recuperação com embeddings reais após a janela de seis turnos.
